# Supplied workbook: reproducible scope audit

## tl;dr

The workbook has 1,067,371 invoice lines across two overlapping worksheets. 22.77% lack customer IDs. The conservative purchase scenario contains 5,852 customers and 4,179 repeat purchase-day customers. It supports the planned project after careful cleaning and temporal validation.

This is a data-quality companion, not a predictive-model experiment.

## Context & Methods

The controlling input is online_retail_II.xlsx, with every worksheet read. The exact profiling implementation is scripts/audit_dataset.py.

### Key Assumptions

- Dates remain in their source-local, timezone-naive form.
- Forecast intervals are end-exclusive and must be completely observed.
- Global exact deduplication and a numeric stock-code pattern define a conservative sensitivity scenario, not final cleaning.
- No upstream event log proves recording completeness.

All code cells were executed sequentially as plain Python by scripts/build_audit_notebook.py. A Jupyter kernel is not used by this generator; the cells are ordinary Python and can also run in Jupyter.

In [1]:
from pathlib import Path
import json
import runpy
import pandas as pd

root = Path.cwd().resolve()
if not (root / 'scripts' / 'audit_dataset.py').exists():
    root = root.parent
assert (root / 'online_retail_II.xlsx').exists(), 'Run from the project root or notebooks folder.'
print('Input: online_retail_II.xlsx; all worksheets; unchanged source file.')

Input: online_retail_II.xlsx; all worksheets; unchanged source file.


## Data

### 1. Recompute the full workbook audit

This cell reads the workbook and regenerates the aggregate evidence. No model is fitted. The script performs null, duplicate, key consistency, sign, code, temporal and forecast-maturity checks.

In [2]:
audit_module = runpy.run_path(str(root / 'scripts' / 'audit_dataset.py'))
profile = audit_module['audit']()
assert sum(sheet['rows'] for sheet in profile['sheets']) == profile['raw_rows']
print('Sheet row counts reconcile with the full raw population.')

Loading every worksheet in the supplied workbook...
Loaded Year 2009-2010: 525,461 invoice lines
Loaded Year 2010-2011: 541,910 invoice lines
{
  "raw_rows": 1067371,
  "date_min": "2009-12-01 07:45:00",
  "date_max": "2011-12-09 12:50:00",
  "distinct_known_customers": 5942,
  "exact_duplicate_excess_rows": 34335
}
Aggregate evidence saved under research/evidence
Sheet row counts reconcile with the full raw population.


## Results

### 2. Review worksheet coverage and quality issues

In [3]:
print(pd.DataFrame(profile['sheets'])[['sheet', 'rows', 'date_min', 'date_max', 'missing_customer_rows']].to_string(index=False))
checks = {
    'Raw invoice lines': profile['raw_rows'],
    'Missing customer lines': profile['nulls']['customer_id']['rows'],
    'Missing customer share (%)': round(profile['nulls']['customer_id']['pct'], 2),
    'Exact duplicate excess lines': profile['exact_duplicate_excess_rows'],
    'Cross-sheet repeated record hashes': profile['cross_sheet_duplicate_hashes'],
    'Negative quantity lines': profile['negative_quantity_rows'],
    'Nonpositive price lines': profile['nonpositive_price_rows'],
}
print(pd.Series(checks, name='Measured value').to_string())

         sheet   rows            date_min            date_max  missing_customer_rows
Year 2009-2010 525461 2009-12-01 07:45:00 2010-12-09 20:01:00                 107927
Year 2010-2011 541910 2010-12-01 08:26:00 2011-12-09 12:50:00                 135080
Raw invoice lines                     1067371.00
Missing customer lines                 243007.00
Missing customer share (%)                 22.77
Exact duplicate excess lines            34335.00
Cross-sheet repeated record hashes      22202.00
Negative quantity lines                 22950.00
Nonpositive price lines                  6207.00


The sheets overlap in early December 2010. Missing IDs affect customer-level coverage. Credits and special codes require separate policies; the table does not classify every such event as an error.

### 3. Check the conservative purchase sample

In [4]:
scenario = profile['provisional_purchase_scenario']
keys = ['candidate_product_rows', 'orders', 'customers', 'one_order_customers', 'repeat_order_customers', 'one_purchase_day_customers', 'repeat_purchase_day_customers']
print(pd.Series({key: scenario[key] for key in keys}).to_string())
assert scenario['one_order_customers'] + scenario['repeat_order_customers'] == scenario['customers']
assert scenario['one_purchase_day_customers'] + scenario['repeat_purchase_day_customers'] == scenario['customers']

candidate_product_rows           776577
orders                            36594
customers                          5852
one_order_customers                1618
repeat_order_customers             4234
one_purchase_day_customers         1673
repeat_purchase_day_customers      4179


These counts describe the audit sensitivity scenario. Final source reconciliation and product classification can change them. Repeat purchase-day customers, rather than repeat invoice customers, determine daily Gamma-Gamma fitting eligibility.

### 4. Check complete forecast horizons

In [5]:
snapshots = pd.DataFrame(profile['snapshots'])
columns = ['cutoff', 'horizon_days', 'label_end_exclusive', 'fully_observed', 'historical_customers', 'inactivity_pct_all_history']
print(snapshots[columns].round({'inactivity_pct_all_history': 2}).to_string(index=False))
assert snapshots.loc[~snapshots['fully_observed'], 'inactivity_pct_all_history'].isna().all()
assert snapshots.loc[snapshots['fully_observed'], 'inactivity_pct_all_history'].between(0, 100).all()
print('Incomplete labels stay missing; observed inactivity percentages have valid bounds.')

    cutoff  horizon_days label_end_exclusive  fully_observed  historical_customers  inactivity_pct_all_history
2010-06-01            90          2010-08-30            True                  2684                       50.30
2010-06-01           180          2010-11-28            True                  2684                       28.69
2010-09-01            90          2010-11-30            True                  3299                       41.68
2010-09-01           180          2011-02-28            True                  3299                       35.34
2010-12-01            90          2011-03-01            True                  4239                       66.81
2010-12-01           180          2011-05-30            True                  4239                       51.88
2011-03-01            90          2011-05-30            True                  4512                       64.87
2011-03-01           180          2011-08-28            True                  4512                       53.66
2

The 90-day final test can begin in September 2011. The six-calendar-month final test can begin in June 2011 and end before December 2011; the 180-day diagnostic has slightly different boundaries. Final-period outcome availability does not prove predictive accuracy, and outcome rates mix cohort composition with seasonality.

## Takeaways

The workbook supports the planned historical customer intelligence project. Reconcile the worksheet overlap, preserve signed credits and anonymous sales, and enforce cutoff/target boundaries before fitting models. Treat forecast revenue, permanent churn and intervention profit as distinct quantities. See research/DATASET_ASSESSMENT.md and research/PROJECT_DESIGN.md for interpretation and the proposed architecture.